In [1]:
!pip install -q -U langchain langgraph langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.0/148.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.6/79.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.5/571.5 kB 16.6 MB/s eta 0:00:00


In [1]:
from google.colab import userdata

GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")

print("API key loaded:", GOOGLE_API_KEY is not None)

API key loaded: True


In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    google_api_key=GOOGLE_API_KEY
)

response = llm.invoke(
    "Explain what an AI agent is in one simple sentence."
)

print(response.content)

[{'type': 'text', 'text': 'An AI agent is a smart software program that can observe its environment, make decisions, and take actions on its own to achieve a specific goal.', 'extras': {'signature': 'EvMNCvANARFNMg/4xAobYYL7IuEGjuAQDTfskBYqDFSAuFyA0qqnbnAOeCqgr5QybRrGfU5014DJwztBWyGnqI5sKm+1sGIkq0rx05VxXSAWEoQI0TmM5SvjI+4qzhcz2qRjDwz2aeyTQ4SrJQn7PwA3QNcYTOPgVOUFUjhqftGR1cL2Og+O0AEgYazjQWLIYEL4e5eoceO0yh5Pgpo9qnH0xsF+waRLYEFX9UGoZwOIq9RPriLKSvOxGX5DYsoKbH+SVxxdaAon7tnd/xLps8Mu/CPml4nEExlx31wCL8dV+oPPu2YT7hfg7miiKsmJg6Ipe6vF98wjTOf1CA64UWSp4T1Ce+p1NYK/byqFvFoet4viZzKaF5L2soFfelUtwRyR0rvBpMiWXwxIt4ddQc6Jzf4smENiwvdWwMBYoLG+8FrHgfHnklUJTrDZcAp1nfzdsIubNZjuwAhwmO7wSegHz31SMJH17S6lyz4znvi0yBB9Uu4o9RHzM/YgIjmgpO5r1OPphPbay9ssZdBHHLKo4b3v4IjKvF+IGjzSB+rsKEcPbq8UXnOzvf6hy0LROQWbvJ1XIXB208LcSAhY4Sz0O9/zcDC5BsHrXn0yEKmcV8sxP5OnIDITFf7AlUnkOEoTLfXSCw8Y8ytnzXuBNEj235iIw/3j9lJvzZTTb2Cnf0sktuji5boDxYfBYIY0XzPN0y/tEkldK55vsnY5LKrFfQdLtJcnlbwnTAOogFsy+gyi3IiA66r3HjJnuXwLuE21WJIf4os1V3t+GbaqiJoThb3iscWk

In [47]:
from typing import TypedDict


class ResearchState(TypedDict):
    question: str
    research_plan: str
    research_results: list
    analysis: str
    fact_check: str
    final_report: str
    research_attempts: int

In [7]:
from langchain_core.prompts import ChatPromptTemplate

planner_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a research planning agent.

Your job is to break a user's research question into
5 specific research tasks.

Do NOT answer the question.
Only create a clear research plan.

Return exactly 5 numbered research tasks."""
    ),
    ("human", "{question}")
])

planner_chain = planner_prompt | llm

In [8]:
question = "What are the effects of artificial intelligence on education?"

response = planner_chain.invoke({
    "question": question
})

print(response.content)

[{'type': 'text', 'text': 'Here is a 5-step research plan to investigate the effects of artificial intelligence on education:\n\n1. **Investigate the impact on personalized learning and student performance:** Analyze academic studies on how AI-driven adaptive learning platforms and intelligent tutoring systems affect student engagement, retention rates, and test scores compared to traditional instruction.\n\n2. **Analyze the influence on teaching practices and administrative efficiency:** Research how teachers utilize AI tools for lesson planning, automated grading, and administrative tasks, and assess the resulting shifts in teacher workload and instructional time.\n\n3. **Examine ethical challenges, academic integrity, and policy responses:** Evaluate the prevalence of generative AI in student work, the efficacy of AI detection tools, issues of data privacy, and how educational institutions are developing honor codes and usage policies.\n\n4. **Assess accessibility, equity, and the d

In [9]:
from typing import TypedDict, List
from langgraph.graph import StateGraph, START, END


class ResearchState(TypedDict):
    question: str
    research_plan: str
    research_results: List[str]
    analysis: str
    fact_check: str
    final_report: str

In [10]:
def planner_node(state: ResearchState):
    response = planner_chain.invoke({
        "question": state["question"]
    })

    # Gemini may return content as a list of blocks
    if isinstance(response.content, list):
        text = "\n".join(
            block.get("text", "")
            for block in response.content
            if isinstance(block, dict) and block.get("text")
        )
    else:
        text = response.content

    return {
        "research_plan": text
    }

In [11]:
graph = StateGraph(ResearchState)

graph.add_node("planner", planner_node)

graph.add_edge(START, "planner")
graph.add_edge("planner", END)

research_graph = graph.compile()

In [12]:
result = research_graph.invoke({
    "question": "What are the effects of artificial intelligence on education?",
    "research_plan": "",
    "research_results": [],
    "analysis": "",
    "fact_check": "",
    "final_report": ""
})

print(result["research_plan"])

Here is a 5-step research plan to analyze the effects of artificial intelligence on education:

1. **Investigate Student Learning Outcomes:** Examine the impact of AI-driven personalized learning platforms and intelligent tutoring systems on student engagement, retention rates, and academic performance.
2. **Assess Educator Workload and Support:** Analyze how AI tools affect teachers' roles, specifically focusing on automated grading, lesson planning, administrative efficiency, and overall teacher workload.
3. **Examine Academic Integrity and Ethical Challenges:** Research the ethical concerns introduced by generative AI in academic settings, including plagiarism, data privacy risks, algorithmic bias, and the reliability of AI detection tools.
4. **Evaluate Accessibility and Educational Equity:** Study the role of AI in improving accessibility for neurodivergent students and learners with disabilities, while assessing the potential risk of worsening the digital divide.
5. **Review Inst

In [13]:
!pip install -q -U tavily-python langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 77.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [14]:
from google.colab import userdata
from tavily import TavilyClient

TAVILY_API_KEY = userdata.get("TAVILY_API_KEY")

tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

print("Tavily connected successfully!")

Tavily connected successfully!


In [15]:
results = tavily_client.search(
    query="latest applications of artificial intelligence in education",
    max_results=3
)

for result in results["results"]:
    print("TITLE:", result["title"])
    print("URL:", result["url"])
    print("CONTENT:", result["content"][:500])
    print("-" * 80)

TITLE: Examples of Artificial Intelligence in Education - Current Applications
URL: https://emerj.com/examples-of-artificial-intelligence-in-education
CONTENT: Out of those provided, intelligent tutoring systems (ITS) seem to have made the most progress over the last 20 years, as one of the original concepts for applications of AI in education. All have the potential to help shape a next generation of more personalized learning and responsive teaching. Our CEO Daniel dives deeper into the near-term future of Online Education and AI on an episode of our podcast, AI in Industry. [...] Though yet to become a standard in schools, artificial intelligence in
--------------------------------------------------------------------------------
TITLE: A systematic review of artificial intelligence applications in ...
URL: https://www.sciencedirect.com/science/article/pii/S277266222500027X
CONTENT: The academic world is becoming increasingly interested in the applications of Artificial Intelligence 

In [16]:
from langchain_core.tools import tool

@tool
def web_search(query: str) -> str:
    """Search the web for reliable information about a research topic."""

    results = tavily_client.search(
        query=query,
        max_results=3
    )

    output = []

    for result in results["results"]:
        output.append(
            f"TITLE: {result['title']}\n"
            f"URL: {result['url']}\n"
            f"CONTENT: {result['content']}\n"
        )

    return "\n---\n".join(output)

In [17]:
researcher_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a research agent.

You receive a research plan created by another agent.

Your job is to:
1. Identify the most important research topics from the plan.
2. Create useful web search queries.
3. Use the web search tool to find relevant information.
4. Focus on trustworthy and recent sources.
5. Return the findings with the source title, URL, and important information.

Do not write the final report.
Your job is only to gather research evidence."""
    ),
    (
        "human",
        """Research question:
{question}

Research plan:
{research_plan}"""
    )
])

In [48]:
def get_text(response):
    """Convert Gemini response into plain text."""

    content = response.content

    if isinstance(content, str):
        return content

    if isinstance(content, list):
        parts = []

        for block in content:
            if isinstance(block, dict):
                text = block.get("text")
                if text:
                    parts.append(text)

        return "\n".join(parts)

    return str(content)


def researcher_node(state: ResearchState):

    question = state["question"]
    research_plan = state["research_plan"]

    # Count how many times we have researched
    attempts = state.get("research_attempts", 0)

    # Ask Gemini to create 3 search queries
    query_prompt = ChatPromptTemplate.from_messages([
        (
            "system",
            """You are a web research query generator.

Given a research question and research plan, create exactly
3 useful web search queries.

The queries should cover different aspects of the topic.

Return ONLY the 3 queries, one per line.
Do not number them.
Do not add explanations."""
        ),
        (
            "human",
            """Research question:
{question}

Research plan:
{research_plan}"""
        )
    ])

    query_chain = query_prompt | llm

    query_response = query_chain.invoke({
        "question": question,
        "research_plan": research_plan
    })

    queries_text = get_text(query_response)

    print("Generated search queries:")
    print(queries_text)

    # Convert response into individual queries
    queries = [
        q.strip()
        for q in queries_text.split("\n")
        if q.strip()
    ]

    queries = queries[:3]

    # Search using Tavily
    research_results = []

    for query in queries:

        print(f"\nSearching: {query}")

        results = tavily_client.search(
            query=query,
            max_results=3
        )

        formatted_results = []

        for result in results["results"]:

            formatted_results.append(
                f"TITLE: {result.get('title', '')}\n"
                f"URL: {result.get('url', '')}\n"
                f"CONTENT: {result.get('content', '')}"
            )

        research_results.append(
            f"SEARCH QUERY: {query}\n\n"
            + "\n\n---\n\n".join(formatted_results)
        )

    return {
        "research_results": research_results,
        "research_attempts": attempts + 1
    }

In [44]:
graph = StateGraph(ResearchState)

graph.add_node("planner", planner_node)
graph.add_node("researcher", researcher_node)
graph.add_node("analyst", analyst_node)
graph.add_node("fact_checker", fact_checker_node)
graph.add_node("writer", writer_node)

graph.add_edge(START, "planner")
graph.add_edge("planner", "researcher")
graph.add_edge("researcher", "analyst")
graph.add_edge("analyst", "fact_checker")

graph.add_conditional_edges(
    "fact_checker",
    check_fact_quality,
    {
        "researcher": "researcher",
        "writer": "writer"
    }
)

graph.add_edge("writer", END)

research_graph = graph.compile()

In [24]:
result = research_graph.invoke({
    "question": "What are the effects of artificial intelligence on education?",
    "research_plan": "",
    "research_results": [],
    "analysis": "",
    "fact_check": "",
    "final_report": ""
})

Generated search queries:
impact of artificial intelligence on student learning outcomes and engagement
effects of AI automation on teacher workload and instructional roles
generative AI in education ethics academic integrity and equity issues

Searching: impact of artificial intelligence on student learning outcomes and engagement

Searching: effects of AI automation on teacher workload and instructional roles

Searching: generative AI in education ethics academic integrity and equity issues


In [25]:
analyst_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are an expert research analyst.

You will receive:
1. A research question
2. A research plan
3. Raw web research collected by another agent

Your job is to analyze the research and identify the most important
findings.

For each major finding:
- Explain the finding clearly.
- Compare information from different sources when possible.
- Mention important evidence.
- Identify disagreements or limitations between sources.
- Do not invent facts.
- Keep the analysis organized and concise.

Do NOT write the final report yet.
Your job is only to analyze the evidence."""
    ),
    (
        "human",
        """Research Question:
{question}

Research Plan:
{research_plan}

Raw Research:
{research_results}"""
    )
])

In [26]:
def analyst_node(state: ResearchState):

    question = state["question"]
    research_plan = state["research_plan"]
    research_results = state["research_results"]

    # Combine all research results into one text
    combined_research = "\n\n".join(
        research_results
    )

    prompt = analyst_prompt.invoke({
        "question": question,
        "research_plan": research_plan,
        "research_results": combined_research
    })

    response = llm.invoke(prompt)

    analysis = get_text(response)

    return {
        "analysis": analysis
    }

In [28]:
result = research_graph.invoke({
    "question": "What are the effects of artificial intelligence on education?",
    "research_plan": "",
    "research_results": [],
    "analysis": "",
    "fact_check": "",
    "final_report": ""
})

Generated search queries:
AI adaptive learning intelligent tutoring impact student outcomes teacher workload
generative AI academic integrity assessment shifts education
AI in education equity digital divide data privacy ethical risks

Searching: AI adaptive learning intelligent tutoring impact student outcomes teacher workload

Searching: generative AI academic integrity assessment shifts education

Searching: AI in education equity digital divide data privacy ethical risks


In [29]:
print("========== ANALYSIS ==========\n")
print(result["analysis"])

========== ANALYSIS ==========

Here is an analysis of the provided research on the effects of artificial intelligence on education, structured around the key areas identified in the research plan.

---

### 1. Personalized Learning and Student Outcomes

* **Measurable Gains in Performance and Engagement:** AI-driven Intelligent Tutoring Systems (ITS) and adaptive learning platforms consistently demonstrate positive impacts on student achievement, engagement, and retention. These systems personalize educational content by analyzing real-time student behavior and performance data to deliver targeted interventions.
* **Target Audience Efficacy:** 
  * *K-12 and Low Prior Knowledge:* Evidence indicates that younger K-12 learners and students with less prior domain knowledge benefit significantly from adaptive platforms due to tailored instruction and structured, repeated practice (NHSJS, citing VanLehn 2011).
  * *Underserved Communities:* High-dose, AI-enhanced tutoring significantly imp

In [30]:
fact_checker_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a careful fact-checking research agent.

You will receive:
1. The original research question
2. An analysis produced by another agent
3. The original web research and sources

Your job is to check whether the important claims in the analysis
are supported by the available evidence.

For each important claim:

- State the claim.
- Decide whether it is Supported, Partially Supported, or Not Supported.
- Give a short explanation.
- Mention the relevant source URL when available.
- Do not invent evidence.
- If the evidence is insufficient, clearly say so.

Be objective and critical."""
    ),
    (
        "human",
        """Research Question:
{question}

Analysis:
{analysis}

Original Research:
{research_results}"""
    )
])

In [31]:
def fact_checker_node(state: ResearchState):

    question = state["question"]
    analysis = state["analysis"]
    research_results = state["research_results"]

    combined_research = "\n\n".join(research_results)

    prompt = fact_checker_prompt.invoke({
        "question": question,
        "analysis": analysis,
        "research_results": combined_research
    })

    response = llm.invoke(prompt)

    fact_check = get_text(response)

    return {
        "fact_check": fact_check
    }

In [34]:
result = research_graph.invoke({
    "question": "What are the effects of artificial intelligence on education?",
    "research_plan": "",
    "research_results": [],
    "analysis": "",
    "fact_check": "",
    "final_report": ""
})

Generated search queries:
impact of artificial intelligence on student learning outcomes and teacher workload
AI in education academic integrity ethical challenges equity data privacy
future of education AI curriculum transformation critical thinking skills

Searching: impact of artificial intelligence on student learning outcomes and teacher workload

Searching: AI in education academic integrity ethical challenges equity data privacy

Searching: future of education AI curriculum transformation critical thinking skills


In [35]:
print("========== FACT CHECK ==========\n")
print(result["fact_check"])

========== FACT CHECK ==========

Here is a fact-checking assessment of the claims made in the analysis, evaluated directly against the provided research sources.

---

### Section 1: Reduction of Educator Workload and Administrative Efficiency

1. **Claim:** UK implementations (Oak National Academy and BCoT using tools like Teachermatic and Google Bard) saved teachers 5.0 to 5.1 hours per week on planning, quizzes, and administrative tasks.
   * **Assessment:** **Supported**
   * **Explanation:** The text specifically states Oak National Academy teachers reported saving "up to five hours per week" and BCoT teachers saved on average "5.1 hours per week" using Google Bard and Teachermatic.
   * **Source:** [https://www.unowa.eu/blog/ai-for-teacher-workload-reduction-case-studies](https://www.unowa.eu/blog/ai-for-teacher-workload-reduction-case-studies)

2. **Claim:** AI tools assist in generating front-end teaching assets (rubrics, summaries, worksheets, draft questions) and automate ro

In [36]:
writer_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are an expert academic research report writer.

Write a clear, well-structured research report using the research
evidence, analysis, and fact-checking results provided.

The report must contain:

1. Title
2. Introduction
3. Key Findings
4. Detailed Analysis
5. Ethical and Social Considerations
6. Limitations
7. Conclusion
8. Sources

Important rules:
- Use only information supported by the provided research.
- Do not invent statistics, studies, or sources.
- Clearly distinguish evidence from interpretation.
- Use a professional academic tone.
- Keep the report readable and well organized.
- Include source URLs where appropriate.
- Do not mention that you are an AI agent.
"""
    ),
    (
        "human",
        """Research Question:
{question}

Research Plan:
{research_plan}

Research Evidence:
{research_results}

Analysis:
{analysis}

Fact Check:
{fact_check}

Write the final research report."""
    )
])

In [37]:
def writer_node(state: ResearchState):

    question = state["question"]
    research_plan = state["research_plan"]
    research_results = state["research_results"]
    analysis = state["analysis"]
    fact_check = state["fact_check"]

    combined_research = "\n\n".join(research_results)

    prompt = writer_prompt.invoke({
        "question": question,
        "research_plan": research_plan,
        "research_results": combined_research,
        "analysis": analysis,
        "fact_check": fact_check
    })

    response = llm.invoke(prompt)

    final_report = get_text(response)

    return {
        "final_report": final_report
    }

In [41]:
result = research_graph.invoke({
    "question": "What are the effects of artificial intelligence on education?",
    "research_plan": "",
    "research_results": [],
    "analysis": "",
    "fact_check": "",
    "final_report": ""
})

Generated search queries:
AI personalized learning systems impact on student performance engagement
effects of generative AI on teacher workload and administrative efficiency
academic integrity ethical concerns AI in education digital divide

Searching: AI personalized learning systems impact on student performance engagement

Searching: effects of generative AI on teacher workload and administrative efficiency

Searching: academic integrity ethical concerns AI in education digital divide


GoogleRateLimitError: Error calling model 'gemini-3.6-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash\nPlease retry in 7.386747092s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'q

In [42]:
print(result["final_report"])

In [43]:
def check_fact_quality(state: ResearchState):

    fact_check = state["fact_check"].lower()

    if "not supported" in fact_check or "partially supported" in fact_check:
        return "researcher"

    return "writer"

In [45]:
result = research_graph.invoke({
    "question": "What are the effects of artificial intelligence on education?",
    "research_plan": "",
    "research_results": [],
    "analysis": "",
    "fact_check": "",
    "final_report": ""
})

print(result["final_report"])

GoogleRateLimitError: Error calling model 'gemini-3.6-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash\nPlease retry in 40.412576829s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', '

In [54]:
def check_fact_quality(state: ResearchState):

    print("Fact check completed → moving to writer.")

    return "writer"

In [50]:
graph = StateGraph(ResearchState)

graph.add_node("planner", planner_node)
graph.add_node("researcher", researcher_node)
graph.add_node("analyst", analyst_node)
graph.add_node("fact_checker", fact_checker_node)
graph.add_node("writer", writer_node)

graph.add_edge(START, "planner")
graph.add_edge("planner", "researcher")
graph.add_edge("researcher", "analyst")
graph.add_edge("analyst", "fact_checker")

graph.add_conditional_edges(
    "fact_checker",
    check_fact_quality,
    {
        "researcher": "researcher",
        "writer": "writer"
    }
)

graph.add_edge("writer", END)

research_graph = graph.compile()

print("Graph compiled successfully!")

Graph compiled successfully!


In [51]:
result = research_graph.invoke({
    "question": "What are the effects of artificial intelligence on education?",
    "research_plan": "",
    "research_results": [],
    "analysis": "",
    "fact_check": "",
    "final_report": "",
    "research_attempts": 0
})

print("Research completed!")

GoogleRateLimitError: Error calling model 'gemini-3.6-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash\nPlease retry in 55.220113058s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', '

In [53]:
print(result["final_report"])

In [55]:
graph = StateGraph(ResearchState)

graph.add_node("planner", planner_node)
graph.add_node("researcher", researcher_node)
graph.add_node("analyst", analyst_node)
graph.add_node("fact_checker", fact_checker_node)
graph.add_node("writer", writer_node)

graph.add_edge(START, "planner")
graph.add_edge("planner", "researcher")
graph.add_edge("researcher", "analyst")
graph.add_edge("analyst", "fact_checker")

graph.add_conditional_edges(
    "fact_checker",
    check_fact_quality,
    {
        "writer": "writer"
    }
)

graph.add_edge("writer", END)

research_graph = graph.compile()

print("Graph compiled successfully!")

Graph compiled successfully!
